In [5]:
import cv2
import mediapipe as mp
import face_recognition
import os
import numpy as np

In [6]:
known_faces_dir = "rostos_conhecidos"
known_encodings = []
known_names = []

for file_name in os.listdir(known_faces_dir):
    path = os.path.join(known_faces_dir, file_name)
    img = face_recognition.load_image_file(path)
    encs = face_recognition.face_encodings(img)
    if encs:
        known_encodings.append(encs[0])
        known_names.append(os.path.splitext(file_name)[0])
print(f"{len(known_encodings)} rostos carregados: {known_names}")

1 rostos carregados: ['igor']


In [7]:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)
drawing = mp.solutions.drawing_utils

def dist(p1, p2):
    return ((p1.x - p2.x)**2 + (p1.y - p2.y)**2)**0.5


In [36]:
cap = cv2.VideoCapture(1)
etapa = "Procurando FACE!"
posicao_inicial_nariz = None
reconhecido = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    small_frame = cv2.resize(frame, (0,0), fx=0.5, fy=0.5)
    rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    resultado = face_mesh.process(rgb_small)
    if "FACE" in etapa and resultado.multi_face_landmarks is not None:
        etapa = "sorriso"
    elif resultado.multi_face_landmarks is None:
        etapa = "Procurando FACE!"
        status = "Procurando FACE!"
        reconhecido = None
    elif "final" == etapa and resultado.multi_face_landmarks is None:
        etapa = "Procurando FACE!"
        reconhecido = None
        status = "Procurando FACE!"
    elif resultado.multi_face_landmarks:
        rosto = resultado.multi_face_landmarks[0]
        nose = rosto.landmark[1]
        le, re = rosto.landmark[33], rosto.landmark[263]
        b1, b2 = rosto.landmark[61], rosto.landmark[291]
        lsup, linf = rosto.landmark[13], rosto.landmark[14]
        d_olhos = dist(le, re)
        largura = dist(b1, b2) / d_olhos
        abertura = dist(lsup, linf) / d_olhos

        if etapa == "sorriso" and largura > 0.6 :
            print("Sorriso detectado!")
            etapa = "girar"

        elif etapa == "girar":
            if posicao_inicial_nariz is None:
                posicao_inicial_nariz = nose.x
            desloc = nose.x - posicao_inicial_nariz
            if abs(desloc) > 0.15:
                print(f"Cabeça girada ({'esq' if desloc < 0 else 'dir'})")
                etapa = "retorno"

        elif etapa == "retorno":
            desloc = nose.x - posicao_inicial_nariz
            if abs(desloc) < 0.05:
                print("Cabeça voltou ao centro")
                etapa = "reconhecimento"

    if etapa == "reconhecimento" and not reconhecido:
        rgb_full = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes = face_recognition.face_locations(rgb_full)
        encs = face_recognition.face_encodings(rgb_full, boxes)

        for enc, box in zip(encs, boxes):
            dists = face_recognition.face_distance(known_encodings, enc)
            for name, d in zip(known_names, dists):
                print(f"→ Distância para {name}: {d:.3f}")
            idx = np.argmin(dists)
            if dists[idx] < 0.5:
                reconhecido = True
                name = known_names[idx]
            else:
                name = "Desconhecido"
            top, right, bottom, left = box
            cv2.rectangle(frame, (left, top), (right, bottom),
                          (0,255,0) if reconhecido else (0,0,255), 2)
            cv2.putText(frame, name, (left, top-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                        (0,255,0) if reconhecido else (0,0,255), 2)
        etapa = "final"

    if etapa == "sorriso":
        status = "Sorria!"
    elif etapa == "girar":
        status = "Vire o rosto"
    elif etapa == "retorno":
        status = "Volte o rosto ao centro"
    elif etapa == "reconhecimento":
        status = "Reconhecendo..."
    elif etapa == "final":
        if reconhecido:
            status = f"{name} autenticado"
        else:
            status = "Nao reconhecido"
    cv2.putText(frame, status, (10, 460),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imshow("Face Auth", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


Sorriso detectado!
Cabeça girada (dir)
Cabeça voltou ao centro
→ Distância para igor: 0.278
Sorriso detectado!
Cabeça girada (esq)
Cabeça voltou ao centro
→ Distância para igor: 0.209
Sorriso detectado!
Cabeça girada (esq)
Cabeça voltou ao centro
→ Distância para igor: 0.212
